# Austin Construction Permit Growth Analysis

This notebook analyzes City of Austin construction permit data to identify where new development is concentrated and to model added square footage as a proxy for growth intensity.

## Project question

1. Which Austin ZIP codes show the highest concentration of new construction?
2. Can permit-level features help predict added square footage (`TotalNewAddSQFT`)?

This is a descriptive and predictive analysis. It is **not** a causal analysis.


## Setup

This notebook expects the source CSV at:

`data/Issued_Construction_Permits.csv`

If your local filename differs, change `DATA_PATH` in the next cell.


In [ ]:
from pathlib import Path

import matplotlib.pyplot as plt
import pandas as pd
from sklearn.compose import ColumnTransformer
from sklearn.ensemble import RandomForestRegressor
from sklearn.impute import SimpleImputer
from sklearn.metrics import mean_absolute_error, r2_score
from sklearn.model_selection import GridSearchCV, train_test_split
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder

pd.set_option("display.max_columns", 200)
pd.set_option("display.max_rows", 200)

DATA_PATH = Path("data/Issued_Construction_Permits.csv")
TARGET_COLUMN = "TotalNewAddSQFT"
FEATURE_COLUMNS = [
    "PermitClass",
    "StatusCurrent",
    "OriginalZip",
    "CouncilDistrict",
    "NumberOfFloors",
    "HousingUnits",
    "TotalJobValuation",
]


## Load data


In [ ]:
if not DATA_PATH.exists():
    raise FileNotFoundError(f"Expected CSV at {DATA_PATH.resolve()}")

raw_df = pd.read_csv(DATA_PATH)
raw_df.shape


In [ ]:
raw_df.head()


## Schema review

This section checks column names, types, missingness, and basic data-quality issues before any analysis.


In [ ]:
raw_df.info()


In [ ]:
quality_summary = pd.DataFrame({
    "dtype": raw_df.dtypes.astype(str),
    "non_null_count": raw_df.notna().sum(),
    "null_count": raw_df.isna().sum(),
    "null_pct": (raw_df.isna().mean() * 100).round(2),
    "n_unique": raw_df.nunique(dropna=True),
}).sort_values("null_pct", ascending=False)

quality_summary.head(25)


## Data cleaning

The analytical target is `TotalNewAddSQFT`. This notebook:

- coerces numeric fields into numeric form
- removes currency symbols from valuation
- filters to positive added square footage
- drops rows missing key geographic fields for mapping


In [ ]:
def clean_currency(series: pd.Series) -> pd.Series:
    return pd.to_numeric(
        series.astype(str).str.replace(r"[\$,]", "", regex=True),
        errors="coerce",
    )


df = raw_df.copy()

df[TARGET_COLUMN] = pd.to_numeric(df[TARGET_COLUMN], errors="coerce")
df["NumberOfFloors"] = pd.to_numeric(df.get("NumberOfFloors"), errors="coerce")
df["HousingUnits"] = pd.to_numeric(df.get("HousingUnits"), errors="coerce")

if "TotalJobValuation" in df.columns:
    df["TotalJobValuation"] = clean_currency(df["TotalJobValuation"])

before_rows = len(df)
df = df.dropna(subset=[TARGET_COLUMN]).copy()
df = df.loc[df[TARGET_COLUMN] > 0].copy()

geo_columns = [c for c in ["OriginalZip", "Latitude", "Longitude"] if c in df.columns]
if geo_columns:
    geo_df = df.dropna(subset=geo_columns).copy()
else:
    geo_df = df.copy()

after_rows = len(df)

pd.DataFrame({
    "stage": ["raw", "cleaned_positive_target"],
    "rows": [before_rows, after_rows],
})


In [ ]:
df[FEATURE_COLUMNS + [TARGET_COLUMN]].head()


## Descriptive summary


In [ ]:
summary = {
    "rows_used": len(df),
    "total_added_sqft": float(df[TARGET_COLUMN].sum()),
    "mean_added_sqft": float(df[TARGET_COLUMN].mean()),
    "median_added_sqft": float(df[TARGET_COLUMN].median()),
}

if "HousingUnits" in df.columns:
    summary["total_housing_units"] = float(df["HousingUnits"].fillna(0).sum())

pd.Series(summary)


## ZIP code concentration

First, inspect where permit activity is concentrated geographically.


In [ ]:
zip_counts = df["OriginalZip"].astype(str).value_counts().head(15)
zip_sqft = (
    df.groupby(df["OriginalZip"].astype(str))[TARGET_COLUMN]
    .sum()
    .sort_values(ascending=False)
    .head(15)
)

zip_counts


In [ ]:
fig, ax = plt.subplots(figsize=(12, 5))
zip_counts.plot(kind="bar", ax=ax)
ax.set_title("Top 15 ZIP Codes by Permit Count")
ax.set_xlabel("ZIP Code")
ax.set_ylabel("Permit Count")
plt.xticks(rotation=45)
plt.tight_layout()


In [ ]:
fig, ax = plt.subplots(figsize=(12, 5))
zip_sqft.plot(kind="bar", ax=ax)
ax.set_title("Top 15 ZIP Codes by Added Square Footage")
ax.set_xlabel("ZIP Code")
ax.set_ylabel("Total Added Square Footage")
plt.xticks(rotation=45)
plt.tight_layout()


## Permit class distribution


In [ ]:
permit_class_counts = df["PermitClass"].fillna("Unknown").value_counts().head(15)
permit_class_counts


In [ ]:
fig, ax = plt.subplots(figsize=(12, 5))
permit_class_counts.plot(kind="bar", ax=ax)
ax.set_title("Top 15 Permit Classes")
ax.set_xlabel("Permit Class")
ax.set_ylabel("Permit Count")
plt.xticks(rotation=45, ha="right")
plt.tight_layout()


## Number of floors and housing units


In [ ]:
df[["NumberOfFloors", "HousingUnits", TARGET_COLUMN]].describe()


In [ ]:
fig, ax = plt.subplots(figsize=(12, 5))
df["NumberOfFloors"].dropna().clip(upper=20).hist(bins=20, ax=ax)
ax.set_title("Distribution of Number of Floors")
ax.set_xlabel("Number of Floors")
ax.set_ylabel("Frequency")
plt.tight_layout()


## Geographic distribution

This scatter plot uses latitude and longitude directly rather than an API-dependent map so the notebook stays reproducible.


In [ ]:
plot_df = geo_df.copy()
if len(plot_df) > 20000:
    plot_df = plot_df.sample(20000, random_state=42)

marker_sizes = (plot_df[TARGET_COLUMN].clip(lower=1) ** (1 / 3)) / 4

fig, ax = plt.subplots(figsize=(10, 9))
ax.scatter(plot_df["Longitude"], plot_df["Latitude"], s=marker_sizes, alpha=0.35)
ax.set_title("Permit Locations Sized by Added Square Footage")
ax.set_xlabel("Longitude")
ax.set_ylabel("Latitude")
plt.tight_layout()


## Modeling target

The target for the predictive task is `TotalNewAddSQFT`.

This is a proxy for construction intensity, not a direct measure of population growth or infrastructure demand.


In [ ]:
model_df = df[FEATURE_COLUMNS + [TARGET_COLUMN]].copy()
model_df.head()


## Train/test split and modeling pipeline


In [ ]:
X = model_df.drop(columns=[TARGET_COLUMN])
y = model_df[TARGET_COLUMN]

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42,
)

numeric_features = X.select_dtypes(include=["number"]).columns.tolist()
categorical_features = [col for col in X.columns if col not in numeric_features]

numeric_transformer = Pipeline(
    steps=[("imputer", SimpleImputer(strategy="median"))]
)

categorical_transformer = Pipeline(
    steps=[
        ("imputer", SimpleImputer(strategy="most_frequent")),
        ("onehot", OneHotEncoder(handle_unknown="ignore")),
    ]
)

preprocessor = ColumnTransformer(
    transformers=[
        ("num", numeric_transformer, numeric_features),
        ("cat", categorical_transformer, categorical_features),
    ]
)

pipeline = Pipeline(
    steps=[
        ("preprocessor", preprocessor),
        ("model", RandomForestRegressor(random_state=42, n_jobs=-1)),
    ]
)

param_grid = {
    "model__n_estimators": [100, 200],
    "model__max_depth": [10, 20, None],
    "model__min_samples_split": [2, 5],
    "model__min_samples_leaf": [1, 2],
}

search = GridSearchCV(
    estimator=pipeline,
    param_grid=param_grid,
    scoring="neg_mean_absolute_error",
    cv=5,
    n_jobs=-1,
    refit=True,
)


In [ ]:
search.fit(X_train, y_train)

train_predictions = search.predict(X_train)
test_predictions = search.predict(X_test)

results = pd.Series({
    "train_mae": mean_absolute_error(y_train, train_predictions),
    "test_mae": mean_absolute_error(y_test, test_predictions),
    "train_r2": r2_score(y_train, train_predictions),
    "test_r2": r2_score(y_test, test_predictions),
})

results


In [ ]:
search.best_params_


## Interpretation

Use the results above conservatively:

- a useful model here means the permit metadata contains predictive signal for added square footage
- it does **not** mean the model directly predicts population, utility load, or gentrification
- permit records are a proxy dataset and should be interpreted as such


## Final takeaways

- Permit activity is concentrated in a limited number of ZIP codes.
- Permit-level metadata can contain useful signal for predicting added square footage.
- Construction permits may provide a faster-moving complement to slower demographic measures.
- This project is best framed as descriptive and predictive support for urban-growth analysis, not as a causal model.
